# Ekstraksi GPU (Kaggle): embedding judul + isi, dan sentimen
Upload `articles.parquet` (dari notebook 00) sebagai Kaggle Dataset, lalu ubah `IN`.
Hasil di `/kaggle/working/cache/` → taruh di `data/cache/` lokal.
**Nama model belum diverifikasi di Hugging Face**, cek dulu kalau error.

In [ ]:
# !pip install -q sentence-transformers
import json, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline

IN = Path('/kaggle/input/geopolitik-articles')      # ganti sesuai nama dataset
OUT = Path('/kaggle/working/cache'); OUT.mkdir(parents=True, exist_ok=True)
EMB_MODEL = 'BAAI/bge-m3'
SENT_MODEL = 'w11wo/indonesian-roberta-base-sentiment-classifier'
MAX_LEN = 4096     # berita rata-rata ~500 token, jadi praktis tanpa chunking

A = pd.read_parquet(IN / 'articles.parquet')
print(len(A))

In [ ]:
m = SentenceTransformer(EMB_MODEL, device='cuda'); m.max_seq_length = MAX_LEN; m.half()
enc = lambda x: m.encode(x, batch_size=8, normalize_embeddings=True, show_progress_bar=True).astype('float32')
np.save(OUT / 'emb_title.npy', enc(A.title.tolist()))       # judul asli
np.save(OUT / 'emb_body.npy', enc(A.text.tolist()))         # isi yang sudah dibersihkan minimal
del m

In [ ]:
clf = pipeline('text-classification', model=SENT_MODEL, device=0, top_k=None)
out = clf((A.title + '. ' + A.text).tolist(), batch_size=32, truncation=True, max_length=512)

def score(o):
    d = {x['label'].lower(): x['score'] for x in o}
    assert {'positive', 'negative'} <= set(d), d     # cek nama label model
    return d['positive'] - d['negative']

np.save(OUT / 'sentiment.npy', np.array([score(o) for o in out], dtype='float32'))

In [ ]:
fp = hashlib.sha256('\n'.join(A.title + '|' + A.url).encode()).hexdigest()[:16]   # sama dengan utils.fingerprint
json.dump(dict(emb_model=EMB_MODEL, sent_model=SENT_MODEL, max_len=MAX_LEN, pooling='bge-m3 default (CLS), L2-norm',
               n=len(A), fingerprint=fp), open(OUT / 'meta.json', 'w'), indent=1)